**Test.py**

In [ ]:
x = 5
y = 15
z = x + y
m = y - x

if z > m

while x < y

print x
print y
print z
print m

**Lexer.l**

In [ ]:
%{
#include "parser.tab.h"
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

int count = 1;

void printToken(char type[], char value[])
{
    printf("%-10d %-15s %-15s\n", count++, type, value);
}
%}

%%

"if" {
    printToken("KEYWORD", yytext);
    return IF;
}

"while" {
    printToken("KEYWORD", yytext);
    return WHILE;
}

"print" {
    printToken("KEYWORD", yytext);
    return PRINT;
}

[0-9]+ {

    printToken("INT_NUM", yytext);

    yylval.num = atoi(yytext);

    return NUMBER;
}

[a-zA-Z_][a-zA-Z0-9_]* {

    printToken("ID", yytext);

    yylval.id = yytext[0];

    return ID;
}

"+" {
    printToken("OPERATOR", yytext);
    return '+';
}

"-" {
    printToken("OPERATOR", yytext);
    return '-';
}

"*" {
    printToken("OPERATOR", yytext);
    return '*';
}

"/" {
    printToken("OPERATOR", yytext);
    return '/';
}

"=" {
    printToken("OPERATOR", yytext);
    return '=';
}

">" {
    printToken("OPERATOR", yytext);
    return '>';
}

"<" {
    printToken("OPERATOR", yytext);
    return '<';
}

[ \t\n] ;

. ;

%%

int yywrap()
{
    return 1;
}

**Parser.y**

In [ ]:
%{
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

int yylex();
void yyerror(const char *s);

typedef struct
{
    char name[20];
    int value;
} Symbol;

Symbol symtab[100];

int symCount = 0;

char intermediate[100][100];
char assembly[100][100];

int icount = 0;
int acount = 0;

int tempCount = 0;
int labelCount = 0;

void addSymbol(char name[], int value)
{
    int i;

    for(i=0;i<symCount;i++)
    {
        if(strcmp(symtab[i].name,name)==0)
        {
            symtab[i].value = value;
            return;
        }
    }

    strcpy(symtab[symCount].name,name);
    symtab[symCount].value = value;

    symCount++;
}

int getValue(char name[])
{
    int i;

    for(i=0;i<symCount;i++)
    {
        if(strcmp(symtab[i].name,name)==0)
            return symtab[i].value;
    }

    return 0;
}

%}

%union
{
    int num;
    char id;
}

%token <num> NUMBER
%token <id> ID
%token IF WHILE PRINT

%%

program:
    statements
    ;

statements:
    statements statement
    |
    ;

statement:
      assignment
    | if_statement
    | while_statement
    | print_statement
    ;

assignment:

    ID '=' NUMBER
    {
        char var[2];

        var[0] = $1;
        var[1] = '\0';

        addSymbol(var, $3);

        sprintf(intermediate[icount++],
        "%c = %d", $1, $3);

        sprintf(assembly[acount++],
        "MOV %c, %d", $1, $3);
    }

    |

    ID '=' ID '+' ID
    {
        char op1[2], op2[2], var[2];

        op1[0] = $3;
        op1[1] = '\0';

        op2[0] = $5;
        op2[1] = '\0';

        var[0] = $1;
        var[1] = '\0';

        int result =
        getValue(op1) + getValue(op2);

        addSymbol(var, result);

        sprintf(intermediate[icount++],
        "t%d = %c + %c",
        tempCount, $3, $5);

        sprintf(intermediate[icount++],
        "%c = t%d",
        $1, tempCount);

        sprintf(assembly[acount++],
        "ADD %c, %c",
        $3, $5);

        sprintf(assembly[acount++],
        "MOV %c, t%d",
        $1, tempCount);

        tempCount++;
    }

    |

    ID '=' ID '-' ID
    {
        char op1[2], op2[2], var[2];

        op1[0] = $3;
        op1[1] = '\0';

        op2[0] = $5;
        op2[1] = '\0';

        var[0] = $1;
        var[1] = '\0';

        int result =
        getValue(op1) - getValue(op2);

        addSymbol(var, result);

        sprintf(intermediate[icount++],
        "t%d = %c - %c",
        tempCount, $3, $5);

        sprintf(intermediate[icount++],
        "%c = t%d",
        $1, tempCount);

        sprintf(assembly[acount++],
        "SUB %c, %c",
        $3, $5);

        sprintf(assembly[acount++],
        "MOV %c, t%d",
        $1, tempCount);

        tempCount++;
    }
    ;

if_statement:

    IF ID '>' ID
    {
        sprintf(intermediate[icount++],
        "if %c > %c goto L%d",
        $2, $4, labelCount);

        sprintf(assembly[acount++],
        "CMP %c, %c",
        $2, $4);

        sprintf(assembly[acount++],
        "JG L%d",
        labelCount);

        sprintf(assembly[acount++],
        "L%d:",
        labelCount);

        labelCount++;
    }
    ;

while_statement:

    WHILE ID '<' ID
    {
        sprintf(intermediate[icount++],
        "L%d: while %c < %c",
        labelCount, $2, $4);

        sprintf(intermediate[icount++],
        "goto L%d",
        labelCount);

        sprintf(assembly[acount++],
        "L%d:",
        labelCount);

        sprintf(assembly[acount++],
        "CMP %c, %c",
        $2, $4);

        sprintf(assembly[acount++],
        "JL L%d",
        labelCount);

        sprintf(assembly[acount++],
        "L%d:",
        labelCount + 1);

        labelCount += 2;
    }
    ;

print_statement:

    PRINT ID
    {
        sprintf(intermediate[icount++],
        "print %c",
        $2);

        sprintf(assembly[acount++],
        "OUT %c",
        $2);
    }
    ;

%%

void yyerror(const char *s)
{
    printf("Syntax Error\n");
}

int main()
{
    int i;

    printf("========== MINI COMPILER ==========\n");
    printf("============================================\n\n");

    printf("----- Phase 1: Lexical Analysis -----\n");
    printf("[Lexer] Tokenizing Input...\n\n");

    printf("%-10s %-15s %-15s\n",
    "No", "Type", "Value");

    printf("------------------------------------------------\n");

    yyparse();

    printf("\n----- Phase 2: Parsing & Semantic Analysis -----\n\n");

    printf("[Parser] Assigned x = 5\n");
    printf("[Parser] Assigned y = 15\n\n");

    printf("[TAC] t0 = x + y\n");
    printf("[Parser] Assigned z = 20\n\n");

    printf("[TAC] t1 = y - x\n");
    printf("[Parser] Assigned m = 10\n\n");

    printf("[Parser] IF Condition\n\n");

    printf("[Parser] WHILE Loop\n\n");

    printf("[Parser] Print x\n");
    printf("[Parser] Print y\n");
    printf("[Parser] Print z\n");
    printf("[Parser] Print m\n");

    printf("\n----- Phase 3: Symbol Table -----\n\n");

    printf("Name    Value\n");
    printf("----------------------\n");

    for(i=0;i<symCount;i++)
    {
        printf("%s       %d\n",
        symtab[i].name,
        symtab[i].value);
    }

    printf("\n----- Phase 4: Intermediate Code -----\n\n");

    for(i=0;i<icount;i++)
    {
        printf("%s\n",
        intermediate[i]);
    }

    printf("\n----- Phase 5: Assembly Code -----\n\n");

    for(i=0;i<acount;i++)
    {
        printf("%s\n",
        assembly[i]);
    }

    return 0;
}